In [111]:
# import libraries
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [112]:
df = pd.read_csv('./datasets/processed/ecommerce_returns_clean.csv')
df_prepared = df.copy()

In [113]:
df_prepared.describe()

,Customer Age,Total Order Value,Discount Applied,Customer Rating,Rating Missing
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,43.516200,12654.593928,0.251241,3.406100,0.080000
std,14.753105,6861.428204,0.144418,1.323086,0.271307
min,18.000000,500.140000,0.000000,1.000000,0.000000
25%,31.000000,6863.377500,0.130000,3.000000,0.000000
50%,43.000000,12708.480000,0.250000,4.000000,0.000000
75%,56.000000,18325.927500,0.380000,4.000000,0.000000
max,69.000000,24994.840000,0.500000,5.000000,1.000000


In [114]:
df_prepared.head(20)

,Order ID,Date,Time of Purchase,Customer Age,Customer Gender,Device Used,Product Category,Total Order Value,Discount Applied,Payment Method,Shipping Type,Customer Rating,Return Status,Rating Missing
0,ORD-100000,2024-10-16,Morning,56,Female,Mobile,Sports,4144.83,0.42,Credit Card,Standard,2.0,Kept,0
1,ORD-100001,2023-04-25,Morning,69,Male,Mobile,Clothing,21436.04,0.01,E-Wallet,Standard,3.0,Returned,0
2,ORD-100002,2023-01-26,Midnight,46,Male,Mobile,Home,15655.16,0.10,Cash on Delivery,Express,4.0,Returned,1
3,ORD-100003,2025-01-29,Evening,32,Female,Desktop,Electronics,12566.34,0.44,E-Wallet,Standard,3.0,Kept,0
4,ORD-100004,2023-10-09,Afternoon,60,Male,Mobile,Home,23072.84,0.49,E-Wallet,Standard,4.0,Kept,0
5,ORD-100005,2023-09-08,Afternoon,25,Male,Desktop,Beauty,6596.37,0.11,Cash on Delivery,Standard,5.0,Kept,0
6,ORD-100006,2023-08-17,Afternoon,38,Female,Desktop,Beauty,6369.90,0.18,Debit Card,Standard,5.0,Kept,0
7,ORD-100007,2023-05-23,Evening,56,Male,Desktop,Beauty,13073.24,0.38,Credit Card,Standard,2.0,Returned,0
8,ORD-100008,2025-01-24,Morning,36,Prefer Not To Say,Mobile,Electronics,18896.15,0.28,Cash on Delivery,Standard,5.0,Returned,0
9,ORD-100009,2023-04-15,Morning,40,Female,Desktop,Sports,11457.40,0.48,Cash on Delivery,Express,4.0,Kept,0


In [115]:
df_prepared.tail(20)

,Order ID,Date,Time of Purchase,Customer Age,Customer Gender,Device Used,Product Category,Total Order Value,Discount Applied,Payment Method,Shipping Type,Customer Rating,Return Status,Rating Missing
9980,ORD-109980,2025-05-18,Morning,31,Prefer Not To Say,Mobile,Clothing,7838.66,0.40,Cash on Delivery,Express,5.0,Kept,0
9981,ORD-109981,2024-08-31,Afternoon,61,Male,Desktop,Beauty,15331.20,0.49,Credit Card,Standard,3.0,Returned,0
9982,ORD-109982,2025-06-11,Afternoon,47,Female,Mobile,Home,6543.08,0.15,E-Wallet,Standard,4.0,Kept,1
9983,ORD-109983,2023-03-31,Evening,34,Female,Mobile,Electronics,15465.68,0.28,Debit Card,Standard,3.0,Kept,0
9984,ORD-109984,2023-12-08,Afternoon,49,Female,Mobile,Clothing,22456.08,0.23,Credit Card,Standard,5.0,Kept,0
9985,ORD-109985,2024-04-23,Afternoon,63,Male,Mobile,Home,14143.48,0.50,E-Wallet,Standard,4.0,Kept,1
9986,ORD-109986,2025-06-29,Evening,23,Female,Desktop,Home,18466.83,0.40,Cash on Delivery,Standard,1.0,Returned,0
9987,ORD-109987,2023-02-17,Midnight,68,Prefer Not To Say,Desktop,Home,12645.57,0.20,E-Wallet,Same-Day,4.0,Kept,0
9988,ORD-109988,2023-04-16,Morning,41,Male,Desktop,Home,13341.04,0.24,Debit Card,Same-Day,2.0,Kept,0
9989,ORD-109989,2024-06-17,Morning,23,Male,Desktop,Electronics,17529.34,0.12,Debit Card,Standard,1.0,Returned,0


In [116]:
df_prepared.columns

Index(['Order ID', 'Date', 'Time of Purchase', 'Customer Age',
       'Customer Gender', 'Device Used', 'Product Category',
       'Total Order Value', 'Discount Applied', 'Payment Method',
       'Shipping Type', 'Customer Rating', 'Return Status', 'Rating Missing'],
      dtype='str')

In [117]:
# snake case column names
df_prepared.columns = df_prepared.columns.str.lower().str.replace(' ', '_')

In [118]:
# drop unnecessary columns
df_prepared.drop('order_id', axis=1, inplace = True)

In [119]:
# engineer date feature
if 'date' in df_prepared.columns:
    df_prepared['date'] = pd.to_datetime(df_prepared['date'])
    df_prepared['order_month'] = df_prepared['date'].dt.month
    df_prepared['order_day'] = df_prepared['date'].dt.day
    df_prepared['is_weekend'] = df_prepared['order_day'].isin([5, 6]).astype(int)
    df_prepared.drop('date', axis=1, inplace=True)

In [120]:
# encode return status
dict_return_status = {'Returned': 1, 'Kept': 0}
if 'return_status' in df_prepared.columns:
    df_prepared['return_status'] = df_prepared['return_status'].map(dict_return_status)

In [121]:
# encode shipping type
dict_shipping_type = {'Standard': 0, 'Express': 1, 'Same-Day': 2}
if 'shipping_type' in df_prepared.columns:
    df_prepared['shipping_type'] = df_prepared['shipping_type'].map(dict_shipping_type)

In [122]:
# encode the rest of categorical features
categorical_columns = ['payment_method', 'product_category', 'device_used', 'customer_gender', 'time_of_purchase']
df_encoded = pd.get_dummies(df_prepared, columns=categorical_columns, drop_first=True, dtype=int)

In [123]:
df_encoded.head(20)

,customer_age,total_order_value,discount_applied,shipping_type,customer_rating,return_status,rating_missing,order_month,order_day,is_weekend,...,product_category_Electronics,product_category_Home,product_category_Sports,device_used_Mobile,device_used_Tablet,customer_gender_Male,customer_gender_Prefer Not To Say,time_of_purchase_Evening,time_of_purchase_Midnight,time_of_purchase_Morning
0,56,4144.83,0.42,0,2.0,0,0,10,16,0,...,0,0,1,1,0,0,0,0,0,1
1,69,21436.04,0.01,0,3.0,1,0,4,25,0,...,0,0,0,1,0,1,0,0,0,1
2,46,15655.16,0.10,1,4.0,1,1,1,26,0,...,0,1,0,1,0,1,0,0,1,0
3,32,12566.34,0.44,0,3.0,0,0,1,29,0,...,1,0,0,0,0,0,0,1,0,0
4,60,23072.84,0.49,0,4.0,0,0,10,9,0,...,0,1,0,1,0,1,0,0,0,0
5,25,6596.37,0.11,0,5.0,0,0,9,8,0,...,0,0,0,0,0,1,0,0,0,0
6,38,6369.90,0.18,0,5.0,0,0,8,17,0,...,0,0,0,0,0,0,0,0,0,0
7,56,13073.24,0.38,0,2.0,1,0,5,23,0,...,0,0,0,0,0,1,0,1,0,0
8,36,18896.15,0.28,0,5.0,1,0,1,24,0,...,1,0,0,1,0,0,1,0,0,1
9,40,11457.40,0.48,1,4.0,0,0,4,15,0,...,0,0,1,0,0,0,0,0,0,1


In [124]:
# separate variables into x and y
X = df_encoded.drop('return_status', axis=1)
y = df_encoded['return_status']

In [125]:
# first split 60%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# second split 40%
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)


In [127]:
print(f"Training set size: {X_train.shape[0]/len(df_encoded): .2f} samples")
print(f"Validation set size: {X_val.shape[0]/len(df_encoded): .2f} samples")
print(f"Test set size: {X_test.shape[0]/len(df_encoded): .2f} samples")

Training set size:  0.60 samples
Validation set size:  0.20 samples
Test set size:  0.20 samples
